Language: **English (default)** | [Polski](lab.ipynb)

# Lab 01 -- multivariate linear regression

Companion material to [`README.md`](README.md). We generalize the
mechanism from lecture 1 (linear regression, gradient, training loop) to
multiple features -- and run into a situation where a linear model isn't
enough.

In [ ]:
import matplotlib.pyplot as plt
import torch

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Synthetic data (given)

Three features $x_1, x_2, x_3$, target $y$. The data is synthetic, with
noise -- this is not part of the task.

In [ ]:
N = 400

x1 = torch.empty(N).uniform_(-3, 3)
x2 = torch.empty(N).uniform_(-3, 3)
x3 = torch.empty(N).uniform_(-3, 3)

w_true = torch.tensor([1.5, -2.0])
quad_coef = 0.8
b_true = 0.5
noise_std = 1.0

y = (
    w_true[0] * x1
    + w_true[1] * x2
    + quad_coef * x3 ** 2
    + b_true
    + torch.randn(N) * noise_std
)

X = torch.stack([x1, x2, x3], dim=1)  # shape (N, 3)

X = X.to(device)
y = y.to(device)

print(f"X.shape={tuple(X.shape)}, y.shape={tuple(y.shape)}, device={X.device}")

## 2. Metrics and data split -- your task

Implement:

- `train_val_split(n_samples, val_fraction)` -- returns `(train_idx, val_idx)`,
  randomly permuting indices `0..n_samples-1` (`torch.randperm`) and
  splitting them in the `val_fraction` proportion for validation,
- `mse(y_pred, y_true)` -- mean squared error,
- `r2_score(y_pred, y_true)` -- $R^2 = 1 - SS_{\text{res}}/SS_{\text{tot}}$.

These three functions are used by the rest of the notebook.

In [ ]:
def train_val_split(n_samples, val_fraction=0.2):
    """Returns (train_idx, val_idx): a random split of 0..n_samples-1."""
    # TODO: draw a random permutation of indices (torch.randperm) and split
    # it into the first `val_fraction * n_samples` (validation) and the
    # rest (training).
    raise NotImplementedError("TODO: train_val_split")


def mse(y_pred, y_true):
    """Mean squared error."""
    # TODO
    raise NotImplementedError("TODO: mse")


def r2_score(y_pred, y_true):
    """R^2 = 1 - SS_res / SS_tot."""
    # TODO
    raise NotImplementedError("TODO: r2_score")

In [ ]:
train_idx, val_idx = train_val_split(N, val_fraction=0.2)
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]

print(f"N_train={len(train_idx)}, N_val={len(val_idx)}")

## 3. Baseline (given)

Prediction = mean $y$ from the training set, independent of $x$.

In [ ]:
baseline_pred_train = torch.full_like(y_train, y_train.mean())
baseline_pred_val = torch.full_like(y_val, y_train.mean())

baseline_mse_val = mse(baseline_pred_val, y_val)
baseline_r2_val = r2_score(baseline_pred_val, y_val)

print(f"Baseline  MSE (val)={baseline_mse_val.item():.4f}  R2 (val)={baseline_r2_val.item():.4f}")

## 4. Model: `forward` -- your task

Generalize `w * x + b` from the lecture to matrix form:
$\hat{y} = Xw + b$, where `X` has shape `(N, 3)`, `w` has shape `(3,)`,
and `b` is a scalar (a tensor of shape `(1,)`).

In [ ]:
def forward(X, w, b):
    """Linear model prediction: X @ w + b."""
    # TODO: matrix product of X and w (operator @ or torch.matmul), plus b.
    raise NotImplementedError("TODO: forward")

## 5. Closed-form solution -- your task

Append a column of ones to `X` (so the bias enters the same system of
equations), then solve the *normal equation*:

$$w^*_{\text{aug}} = (X_{\text{aug}}^\top X_{\text{aug}})^{-1} X_{\text{aug}}^\top y$$

Return `w` (the first 3 coefficients) and `b` (the last one)
**separately**, so they match the `forward(X, w, b)` signature from
section 4. Use `torch.linalg.solve` (solving the linear system) rather
than explicitly inverting the matrix -- it's more numerically stable.

In [ ]:
def closed_form_solution(X, y):
    """Returns (w, b) from the normal equation on the training data."""
    # TODO:
    # 1. append a column of ones to X (torch.cat with torch.ones(N, 1))
    # 2. solve (X_aug^T X_aug) w_aug = X_aug^T y using torch.linalg.solve
    # 3. split w_aug into w (first 3 elements) and b (last element)
    raise NotImplementedError("TODO: closed_form_solution")


w_closed, b_closed = closed_form_solution(X_train, y_train)
closed_form_val_loss = mse(forward(X_val, w_closed, b_closed), y_val)
print(f"Closed-form solution:  w={w_closed.tolist()}  b={b_closed.item():.4f}")
print(f"Loss (val) of the closed-form solution: {closed_form_val_loss.item():.4f}")

## 6. Training via gradient descent -- your task

Generalize the lecture's training loop to vector-valued `w` (shape
`(3,)`). Same principle: *forward -> loss -> backward -> update*, without
`nn.Module`/`optim`.

In [ ]:
learning_rate = 0.05
n_epochs = 300


def train_linear_model(X_train, y_train, learning_rate, n_epochs):
    """Returns (w, b, loss_history) trained via gradient descent."""
    # TODO:
    # 1. initialize w (shape (3,)) and b (shape (1,)) as torch.zeros with
    #    requires_grad=True, on the appropriate device
    # 2. in a loop over n_epochs: forward -> loss (mse) -> loss.backward()
    #    -> update under torch.no_grad() -> zero the gradients
    # 3. collect each epoch's loss into a list and return (w, b, loss_history)
    raise NotImplementedError("TODO: train_linear_model")


w_gd, b_gd, loss_history = train_linear_model(X_train, y_train, learning_rate, n_epochs)
print(f"Gradient descent:  w={w_gd.tolist()}  b={b_gd.item():.4f}")

## 7. Experiment: comparing losses (given)

In [ ]:
gd_val_loss = mse(forward(X_val, w_gd, b_gd), y_val)

print(f"Baseline              loss (val) = {baseline_mse_val.item():.4f}")
print(f"Closed-form solution  loss (val) = {closed_form_val_loss.item():.4f}")
print(f"Gradient descent      loss (val) = {gd_val_loss.item():.4f}")

plt.figure(figsize=(5, 4))
plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("loss (MSE, training)")
plt.title("Loss curve -- gradient descent")
plt.tight_layout()
plt.show()

## 8. Evaluation on the validation set (given)

In [ ]:
y_pred_val = forward(X_val, w_gd, b_gd)

val_mse = mse(y_pred_val, y_val)
val_r2 = r2_score(y_pred_val, y_val)

print(f"Model (GD) on the validation set: MSE={val_mse.item():.4f}  R2={val_r2.item():.4f}")

## 9. Error analysis: residual plot (given) + interpretation -- your task

In [ ]:
residuals_val = (y_pred_val - y_val).detach().cpu()
features_val = X_val.detach().cpu()
feature_names = ["x1", "x2", "x3"]

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for i, ax in enumerate(axes):
    ax.scatter(features_val[:, i], residuals_val, s=15, alpha=0.6)
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.set_xlabel(feature_names[i])
    ax.set_title(f"residuals vs {feature_names[i]}")
axes[0].set_ylabel("residual (ŷ - y)")
plt.tight_layout()
plt.show()

**Task (3-5 sentences, Markdown cell):** which feature has a
systematic, non-random pattern in the residual? What does this pattern
look like? What is the linear model *structurally* unable to express that
causes this pattern?

_(Your answer here.)_

## 10. Conclusions -- your task

Answer (markdown, justified, not one-liners):

1. Could this same problem be solved better by adding a new *feature*
   (e.g. the square of an existing one) without changing the model itself?
   Justify your answer.

   _(Your answer here.)_

2. Why is comparing against the closed-form solution a useful correctness
   test for your training loop?

   _(Your answer here.)_